<a href="https://colab.research.google.com/github/NethmiAmasha/Deep-Learning-for-Vision/blob/main/EN4554_AS02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#EN4554 Assignment 2 on CNNs
----

Name: Pathirana P.N.A

Index No.: 220449F


### Task 1: Backpropagation

(a) Linear, ReLU, and softmax cross-entropy

In [2]:
import torch

class Linear:
    def __init__(self, in_features, out_features, dtype=torch.float32, generator=None):
        std = (2.0 / in_features) ** 0.5   # Kaiming initialization
        self.W = torch.randn(in_features, out_features, dtype=dtype, generator=generator) * std
        self.b = torch.zeros(out_features, dtype=dtype)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW = self.x.T @ dout
        self.db = dout.sum(dim=0)
        return dout @ self.W.T


class ReLU:
    def forward(self, x):
        self.mask = (x > 0).to(x.dtype)
        return x * self.mask

    def backward(self, dout):
        return dout * self.mask


class SoftmaxCrossEntropy:
    def forward(self, logits, y):
        shifted = logits - logits.max(dim=1, keepdim=True).values    # numerical stability
        log_probs = shifted - torch.logsumexp(shifted, dim=1, keepdim=True)
        self.probs = log_probs.exp()
        self.y = y
        N = logits.shape[0]
        return -log_probs[torch.arange(N), y].mean()

    def backward(self):
        N = self.probs.shape[0]
        dlogits = self.probs.clone()
        dlogits[torch.arange(N), self.y] -= 1.0
        return dlogits / N

MLP

In [3]:
class MLP:
    def __init__(self, dims, dtype=torch.float32, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.layers = []
        for i in range(len(dims) - 1):
            self.layers.append(Linear(dims[i], dims[i + 1], dtype=dtype, generator=g))
            if i < len(dims) - 2:
                self.layers.append(ReLU())
        self.loss_fn = SoftmaxCrossEntropy()

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def loss(self, x, y):
        return self.loss_fn.forward(self.forward(x), y)

    def backward(self):
        grad = self.loss_fn.backward()
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    def linear_layers(self):
        return [l for l in self.layers if isinstance(l, Linear)]

(b) Verifying the gradients against autograd and against centred finite differences

In [5]:
import torch
import torch.nn.functional as F

def rel_err(a, b, eps=1e-12):
    return ((a - b).norm() / (a.norm() + b.norm() + eps)).item()

def autograd_reference(model, x, y):
    params = []
    for layer in model.linear_layers():
        W = layer.W.clone().requires_grad_(True)
        b = layer.b.clone().requires_grad_(True)
        params.append((W, b))

    h = x
    for i, (W, b) in enumerate(params):
        h = h @ W + b
        if i < len(params) - 1:
            h = torch.relu(h)
    loss = F.cross_entropy(h, y)    # mean softmax cross-entropy
    loss.backward()
    return loss.item(), [(W.grad, b.grad) for W, b in params]

In [6]:
def finite_diff_check(model, x, y, eps=1e-6, n_samples=25, seed=0):
    g = torch.Generator().manual_seed(seed)
    model.loss(x, y)
    model.backward()   # manual gradients
    results = []
    for li, layer in enumerate(model.linear_layers()):
        for name in ("W", "b"):
            param = getattr(layer, name)
            grad = getattr(layer, "d" + name)
            flat = param.view(-1)
            idx = torch.randperm(flat.numel(), generator=g)[:n_samples]
            ana = grad.reshape(-1)[idx]  # analytic gradient at the sampled entries
            num = torch.empty_like(ana)
            for k, i in enumerate(idx.tolist()):
                orig = flat[i].item()
                flat[i] = orig + eps
                loss_plus = model.loss(x, y).item()
                flat[i] = orig - eps
                loss_minus = model.loss(x, y).item()
                flat[i] = orig
                num[k] = (loss_plus - loss_minus) / (2 * eps)
            results.append((f"layer{li}.{name}",
                            rel_err(ana, num),
                            torch.allclose(ana, num, rtol=1e-5, atol=1e-7)))
    return results

In [7]:
torch.manual_seed(0)
model = MLP([3072, 64, 32, 10], dtype=torch.float64, seed=0)
x = torch.randn(16, 3072, dtype=torch.float64)
y = torch.randint(0, 10, (16,))

# manual vs autograd
loss = model.loss(x, y).item()
model.backward()
ag_loss, ag_grads = autograd_reference(model, x, y)
print(f"loss  manual={loss:.12f}  autograd={ag_loss:.12f}\n")

print("=== Manual vs autograd ===")
for li, (layer, (gW, gb)) in enumerate(zip(model.linear_layers(), ag_grads)):
    for name, mine, ref in (("W", layer.dW, gW), ("b", layer.db, gb)):
        print(f"layer{li}.{name}: rel err = {rel_err(mine, ref):.2e}   "
              f"allclose = {torch.allclose(mine, ref, rtol=1e-7, atol=1e-10)}")

# manual vs finite differences
print("\n=== Manual vs centred finite differences ===")
for name, err, ok in finite_diff_check(model, x, y):
    print(f"{name}: rel err = {err:.2e}   allclose = {ok}")

loss  manual=3.217372143849  autograd=3.217372143849

=== Manual vs autograd ===
layer0.W: rel err = 8.24e-17   allclose = True
layer0.b: rel err = 9.09e-17   allclose = True
layer1.W: rel err = 6.25e-17   allclose = True
layer1.b: rel err = 4.79e-17   allclose = True
layer2.W: rel err = 5.21e-17   allclose = True
layer2.b: rel err = 2.62e-17   allclose = True

=== Manual vs centred finite differences ===
layer0.W: rel err = 2.59e-09   allclose = True
layer0.b: rel err = 1.65e-09   allclose = True
layer1.W: rel err = 1.57e-09   allclose = True
layer1.b: rel err = 1.51e-09   allclose = True
layer2.W: rel err = 9.65e-10   allclose = True
layer2.b: rel err = 9.53e-10   allclose = True


The manual gradients match autograd to ~1e-17 relative error and centred finite differences to ~1e-9, with torch.allclose passing for every parameter. So, the backward pass is correct. The check ran in float64 on a smaller MLP (3072-64-32-10, 16 random inputs), and the finite-difference check tested 25 randomly sampled entries per tensor with ε = 1e-6.